# Train an Explainable xAAEnet Model

> Run adversarial, autoencoder, and classifier training in one call, with configurable loss weights for each phase.

This module wraps the fastai training loop for models that expose the xAAEnet losses (`aae_loss_func`, `denoising_ae_loss_func`, `classif_loss_func`), including `AAE` and `EncoderWithAAEBlocks`.

In [ ]:
#| default_exp training

## Training pipeline

`train_xaaenet` always runs the same sequence:

1. **Adversarial** — latent GAN only, with alternating generator / discriminator updates.
2. **Autoencoder** — denoising reconstruction on corrupted inputs (Gaussian noise + random patch masking).
3. **Classifier** — weighted sum of reconstruction, adversarial, and cross-entropy terms.

Each phase saves a checkpoint under `models_dir`, reloads the best weights then continues to the next phase.

After each phase, when visualization is enabled, `train_xaaenet` follows the same order:

1. **Extract** validation latent vectors `z` from the trained model.
2. **Display** the figure inline in the notebook (when running under IPython).
3. **Save** a PNG under `models_dir`.

- After **adversarial** and **autoencoder**: t-SNE (`save_tsne=True`).
- After **classifier** (default curriculum): supervised **PLS** biplot (`save_pls=True`). Callable after any phase via `_pls_validation_latent_after_phase`.

## Quick start

Build your model and fastai `DataLoaders` (`ImageBlock`, `CategoryBlock`), then call:

```python
from tell_me_why.training import train_xaaenet

learn = train_xaaenet(
    model,
    dls,
    ae_recons_weight=0.4,
    ae_adv_weight=0.6,
    classif_recons_weight=0.599,
    classif_class_weight=0.001,
    classif_adv_weight=0.4,
)
```

Input images should be at least about **160×160** pixels (MS-SSIM in the reconstruction loss).

## `train_xaaenet`

| Argument | Role |
|----------|------|
| `epochs_adv`, `epochs_ae`, `epochs_classif` | Epoch count per phase |
| `ae_recons_weight`, `ae_adv_weight` | Loss weights during autoencoder training |
| `classif_recons_weight`, `classif_class_weight`, `classif_adv_weight` | Loss weights during classifier training |
| `adv_low_threshold`, `adv_high_threshold` | Valid-loss band for alternating GAN training |
| `mask_ratio`, `patch_size`, `noise_std` | Corruption strength for denoising AE training |
| `models_dir`, `*_fname` | Checkpoint directory and file names |
| `save_tsne` | After adversarial / autoencoder: extract validation `z`, show t-SNE, save PNG |
| `tsne_max_points`, `tsne_perplexity` | Subsample size and t-SNE perplexity for those figures |
| `save_pls` | After classifier phase (default): extract validation `z`, show PLS biplot, save PNG |
| `show_latent_figures` | Inline notebook display before PNG save (`None` = auto in IPython) |
| `pls_target_class` | Binary target level in `dls.vocab` for the PLS axis (default: second class) |
| `pls_max_points` | Max validation points for the PLS figure (default: 10 000) |
| `extract_latent` | Save train+valid latent vectors `z` after the full run |

In [ ]:
#| export
#| hide
from __future__ import annotations

from collections.abc import Sequence
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from fastai.callback.tracker import EarlyStoppingCallback, SaveModelCallback
from fastai.callback.training import GradientAccumulation
from fastai.data.all import Transform
from fastai.metrics import Metric
from fastai.vision.all import *
from torch import Tensor

from tell_me_why.model_aae import AAE, default_device
from tell_me_why.user_encoder import EncoderWithAAEBlocks

ExplainableModel = AAE | EncoderWithAAEBlocks

## Corruption transforms (autoencoder phase)

During phase 2, inputs are corrupted before the forward pass; reconstruction is computed against the **clean** batch stored by `CorruptionCallback`.

In [ ]:
#| export
class AddGaussianNoise(Transform):
    """Add Gaussian noise to a batch of images (values stay in [0, 1])."""

    def __init__(self, mean: float = 0.0, std: float = 0.05):
        self.mean, self.std = mean, std

    def encodes(self, x: TensorImage):
        noise = torch.randn_like(x) * self.std + self.mean
        return (x + noise).clamp(0, 1)


class RandomMasking(Transform):
    """Zero out random square patches independently per image."""

    def __init__(self, mask_ratio: float = 0.3, patch_size: int = 16):
        self.mask_ratio = mask_ratio
        self.patch_size = patch_size

    def encodes(self, x: TensorImage):
        x = x.clone()
        _, _, h, w = x.shape
        n_patches_h = h // self.patch_size
        n_patches_w = w // self.patch_size
        n_masked = int((n_patches_h * n_patches_w) * self.mask_ratio)
        for b in range(x.shape[0]):
            indices = torch.randperm(n_patches_h * n_patches_w, device=x.device)[:n_masked]
            for idx in indices:
                i = (idx // n_patches_w) * self.patch_size
                j = (idx % n_patches_w) * self.patch_size
                x[b, :, i : i + self.patch_size, j : j + self.patch_size] = 0.0
        return x


class CorruptionCallback(Callback):
    """Apply corruption transforms on `xb` while keeping a clean copy for the loss."""

    def __init__(self, corruption_tfms: list):
        self.corruption_tfms = corruption_tfms

    def before_batch(self):
        self.learn.clean_xb = self.learn.xb[0].clone()
        corrupted = self.learn.xb[0].clone()
        for tfm in self.corruption_tfms:
            corrupted = tfm(corrupted)
        self.learn.xb = (corrupted,)

## Adversarial callback

`UnfreezeFcCritAdaptative` toggles `model.gen_train` and freezes either the discriminator (`fc_crit*`) or the rest of the network, based on validation loss and epoch schedule.

In [ ]:
#| export
class UnfreezeFcCritAdaptative(Callback):
    """Alternate generator / discriminator training during the adversarial phase."""

    def __init__(
        self,
        switch_every: int = 3,
        low_threshold: float = 0.15,
        high_threshold: float = 0.60,
        window_size: int = 3,
    ):
        self.switch_every = switch_every
        self.low_threshold = low_threshold
        self.high_threshold = high_threshold
        self.window_size = window_size
        self.valid_loss_history: list[float] = []
        self.gen_train_epochs = 0

    def _set_gen_train(self, gen_train: bool) -> None:
        self.learn.model.gen_train = gen_train
        for name, param in self.learn.model.named_parameters():
            param.requires_grad_("fc_crit" not in name if gen_train else "fc_crit" in name)

    def before_epoch(self):
        if self.learn.recorder.values:
            self.valid_loss_history.append(self.learn.recorder.values[-1][1])
        window = self.valid_loss_history[-self.window_size :]
        avg_valid_loss = float(np.mean(window)) if window else float("inf")

        if self.epoch < 3:
            self._set_gen_train(False)
            self.gen_train_epochs = 0
        elif self.epoch < 6:
            self._set_gen_train(True)
            self.gen_train_epochs += 1
        elif avg_valid_loss < self.low_threshold:
            self._set_gen_train(False)
            self.gen_train_epochs = 0
        elif avg_valid_loss > self.high_threshold:
            self._set_gen_train(True)
            self.gen_train_epochs += 1
        elif (self.epoch + 1) % self.switch_every == 0:
            self._set_gen_train(False)
            self.gen_train_epochs = 0
        else:
            self._set_gen_train(True)
            self.gen_train_epochs += 1

        if self.gen_train_epochs >= 3:
            self._set_gen_train(False)
            self.gen_train_epochs = 0

## Metrics, latent extraction, and t-SNE figures

`LossAttrMetric` logs scalar attributes set on the model during the forward pass (`adv_loss`, `recons_loss`, …). Latent extraction for figures and `extract_latent=True` runs a validation pass and stacks `model.z` — you do not need to call a separate callback yourself.

At the end of each training phase, `train_xaaenet` **extracts** validation `z`, then calls:

- `save_latent_tsne_figure` — t-SNE (default: up to 5000 points, perplexity 30), **display** inline, then PNG.
- `save_latent_pls_figure` — supervised **PLS** (2 components) on `z` and binary targets, **display** inline, then PNG (one arrow for the target direction).

## Example output figures

With default options, figures appear in the notebook then are written as PNG under `models_dir`. Below are representative examples (validation set, binary classification).

### t-SNE (`save_tsne=True`)

Saved after the **adversarial** and **autoencoder** phases as `tsne_<checkpoint_name>.png`:

![t-SNE of the validation latent space](images/tsne_example.png)

Direct projection of `z` (e.g. 128D → 2D). Points are not colored by class; the plot monitors how the latent space evolves between phases.

### PLS (`save_pls=True`)

Saved once after the **classifier** phase as `pls_<classif_fname>.png`:

![PLS biplot — supervised latent space](images/pls_example.png)

Supervised PLS on `z`: color = score on PLS component 1, **one white arrow** = direction of the binary target in latent space.

The figure above illustrates the overall PLS style (axes, colorbar, latent space).

In [ ]:
#| export
class GetLatentSpace(Callback):
    """Collect latent vectors `z` during validation into `learn.z_valid`."""

    def before_validate(self):
        self.learn.z_valid = torch.tensor([]).to(self.learn.model.z.device)

    def after_batch(self):
        if not self.training:
            z = self.learn.model.z.detach()
            if self.learn.z_valid.numel() == 0:
                self.learn.z_valid = z
            else:
                self.learn.z_valid = torch.vstack((self.learn.z_valid, z))


class LossAttrMetric(Metric):
    """Average a float attribute stored on the model (e.g. `model.adv_loss`)."""

    def __init__(self, attr: str):
        self.attr_name = attr
        self.vals: list[float] = []

    def reset(self):
        self.vals = []

    def accumulate(self, learn):
        val = getattr(learn.model, self.attr_name)
        if hasattr(val, "item"):
            val = val.item()
        self.vals.append(float(val))

    @property
    def value(self):
        return torch.tensor(self.vals).mean() if self.vals else torch.tensor(0.0)

    @property
    def name(self):
        return self.attr_name


def _extract_latent_and_targets(learn: Learner, ds_idx: int = 1) -> tuple[torch.Tensor, torch.Tensor]:
    """Return stacked latent vectors `z` and class indices for a dataloader split."""
    learn.model.eval()
    learn.z_valid = torch.tensor([]).to(default_device())
    with torch.no_grad():
        _, targs = learn.get_preds(ds_idx=ds_idx, cbs=[GetLatentSpace()])
    return learn.z_valid.cpu(), targs.cpu().view(-1).long()


def _extract_latent_vectors(learn: Learner, ds_idx: int = 1) -> torch.Tensor:
    """Run inference on a dataloader split and return stacked latent vectors `z`."""
    z, _ = _extract_latent_and_targets(learn, ds_idx)
    return z


def _display_figure(fig, *, show: bool | None = None) -> None:
    """Show *fig* inline in IPython (notebook); no-op in plain scripts unless ``show=True``."""
    if show is False:
        return
    try:
        from IPython import get_ipython
        from IPython.display import display
    except ImportError:
        return
    if show is True or get_ipython() is not None:
        display(fig)


def save_latent_tsne_figure(
    z: torch.Tensor,
    save_path: str | Path,
    *,
    phase: str,
    encoding_dims: int | None = None,
    max_points: int = 5000,
    perplexity: float = 30,
    random_state: int = 42,
    show: bool | None = None,
) -> Path:
    """Project ``z`` with t-SNE, display inline in a notebook, then save a PNG."""
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE

    X = z.numpy() if isinstance(z, torch.Tensor) else np.asarray(z)
    n_total = len(X)
    if n_total > max_points:
        idx = np.random.default_rng(random_state).choice(n_total, max_points, replace=False)
        X = X[idx]
    n = len(X)
    enc = encoding_dims or X.shape[1]
    perp = min(perplexity, max(5, n - 1))
    X2 = TSNE(n_components=2, perplexity=perp, init="random", random_state=random_state).fit_transform(X)

    fig, ax = plt.subplots(figsize=(10, 10), facecolor="#0d1117")
    ax.set_facecolor("#0d1117")
    ax.scatter(X2[:, 0], X2[:, 1], s=4, c="#6eb5ff", alpha=0.55, linewidths=0)
    ax.set_title(
        f"t-SNE direct — espace latent {enc}D → 2D\n"
        f"N={n} points · perplexité={perp:.0f} · {phase}",
        color="white",
        fontsize=13,
    )
    ax.set_xlabel("Dimension t-SNE 1", color="white")
    ax.set_ylabel("Dimension t-SNE 2", color="white")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_color("#444")
    ax.grid(True, alpha=0.15, color="#666")
    fig.tight_layout()
    _display_figure(fig, show=show)
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, facecolor=fig.get_facecolor(), bbox_inches="tight")
    plt.close(fig)
    print(f"t-SNE saved: {save_path}")
    return save_path


def save_latent_pls_figure(
    z: torch.Tensor,
    targets: torch.Tensor,
    save_path: str | Path,
    *,
    class_names: Sequence[str] | None = None,
    target_class: str | None = None,
    phase: str = "",
    encoding_dims: int | None = None,
    max_points: int = 10_000,
    random_state: int = 42,
    show: bool | None = None,
) -> Path:
    """Supervised PLS biplot of ``z`` vs binary targets: display inline in a notebook, then save a PNG."""
    import matplotlib.pyplot as plt
    import scipy.stats as stats
    from sklearn.cross_decomposition import PLSRegression

    Z = z.numpy() if isinstance(z, torch.Tensor) else np.asarray(z, dtype=np.float64)
    y_idx = targets.numpy() if isinstance(targets, torch.Tensor) else np.asarray(targets).reshape(-1)
    names = list(class_names) if class_names is not None else ["0", "1"]
    if len(names) != 2:
        raise ValueError("save_latent_pls_figure expects exactly two class names (binary targets).")
    pos_idx = names.index(target_class) if target_class is not None else 1
    y_score = (y_idx == pos_idx).astype(np.float64)

    n_total = len(Z)
    if n_total > max_points:
        idx = np.random.default_rng(random_state).choice(n_total, max_points, replace=False)
        Z, y_score = Z[idx], y_score[idx]

    pls = PLSRegression(n_components=2)
    pls.fit(Z, y_score.reshape(-1, 1))
    z_pls = pls.transform(Z)
    r2_c1 = stats.pearsonr(z_pls[:, 0], y_score)[0] ** 2
    r2_c2 = stats.pearsonr(z_pls[:, 1], y_score)[0] ** 2
    target_label = names[pos_idx]
    enc = encoding_dims or Z.shape[1]

    fig, ax = plt.subplots(figsize=(12, 10), facecolor="#161b22")
    ax.set_facecolor("#0e1117")
    scatter = ax.scatter(
        z_pls[:, 0], z_pls[:, 1], c=z_pls[:, 0], cmap="viridis", alpha=0.7, s=30, edgecolors="none"
    )
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label(f"Intensité estimée (score PLS de '{target_label}')", color="#c9d1d9")
    cbar.ax.yaxis.set_tick_params(color="#8b949e")
    plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="#8b949e")

    r_x, _ = stats.pearsonr(z_pls[:, 0], y_score)
    r_y, _ = stats.pearsonr(z_pls[:, 1], y_score)
    scale = max(np.max(np.abs(z_pls[:, 0])), np.max(np.abs(z_pls[:, 1])), 1e-6) * 0.8
    mx, my = np.mean(z_pls[:, 0]), np.mean(z_pls[:, 1])
    dx, dy = r_x * scale, r_y * scale
    ax.annotate(
        "",
        xy=(mx + dx, my + dy),
        xytext=(mx, my),
        arrowprops=dict(arrowstyle="->", color="#ffffff", lw=2.5),
    )
    ax.text(
        mx + dx * 1.08,
        my + dy * 1.08,
        target_label,
        color="#ffffff",
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        bbox=dict(facecolor="#0e1117", edgecolor="none", alpha=0.7, pad=1),
    )
    ax.axhline(my, color="#30363d", linestyle="--", linewidth=1)
    ax.axvline(mx, color="#30363d", linestyle="--", linewidth=1)
    phase_line = f" · {phase}" if phase else ""
    ax.set_title(
        f"Espace PLS — direction cible '{target_label}'\n"
        f"latent {enc}D → 2D · N={len(Z)} points{phase_line}",
        color="white",
        fontsize=14,
        pad=16,
    )
    ax.set_xlabel(f"PLS 1 — direction '{target_label}'  (r²={r2_c1:.3f})", color="#c9d1d9", fontsize=11)
    ax.set_ylabel(f"PLS 2 — variance orthogonale  (r²={r2_c2:.3f})", color="#c9d1d9", fontsize=11)
    ax.tick_params(colors="#8b949e")
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    ax.grid(True, linestyle=":", color="#30363d", alpha=0.5)
    fig.tight_layout()
    _display_figure(fig, show=show)
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"PLS figure saved: {save_path}")
    return save_path


#| hide
def _tsne_validation_latent_after_phase(
    learn: Learner,
    save_path: str | Path,
    *,
    phase: str,
    encoding_dims: int | None,
    max_points: int,
    perplexity: float,
    show: bool | None,
) -> torch.Tensor:
    """Extract validation ``z``, show t-SNE in the notebook, save PNG; return ``z``."""
    print(f"Latent z — validation ({phase})")
    z_val = _extract_latent_vectors(learn)
    print(f"  shape: {tuple(z_val.shape)}")
    save_latent_tsne_figure(
        z_val,
        save_path,
        phase=phase,
        encoding_dims=encoding_dims,
        max_points=max_points,
        perplexity=perplexity,
        show=show,
    )
    return z_val


def _pls_validation_latent_after_phase(
    learn: Learner,
    save_path: str | Path,
    dls: DataLoaders,
    *,
    phase: str,
    target_class: str | None,
    encoding_dims: int | None,
    max_points: int,
    show: bool | None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Extract validation ``z`` and targets, show PLS in the notebook, save PNG; return ``(z, targets)``."""
    print(f"Latent z — validation ({phase})")
    z_val, targs_val = _extract_latent_and_targets(learn, ds_idx=1)
    print(f"  shape: {tuple(z_val.shape)}")
    save_latent_pls_figure(
        z_val,
        targs_val,
        save_path,
        class_names=tuple(dls.vocab),
        target_class=target_class,
        phase=phase,
        encoding_dims=encoding_dims,
        max_points=max_points,
        show=show,
    )
    return z_val, targs_val

In [ ]:
#| export
#| hide
def _aae_splitter(model: nn.Module):
    backbone = [p for n, p in model.named_parameters() if "unet" in n and "fc_crit" not in n]
    head = [p for n, p in model.named_parameters() if "unet" not in n and "fc_crit" not in n]
    crit = [p for n, p in model.named_parameters() if "fc_crit" in n]
    return [backbone, head, crit]


def _load_checkpoint(model: nn.Module, models_dir: Path, fname: str) -> None:
    path = models_dir / f"{fname}.pth"
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location=default_device(), weights_only=True)
    model.load_state_dict(state, strict=False)


class _AAELoss:
    def __init__(self, model: ExplainableModel):
        self.model = model

    def __call__(self, pred, *yb):
        return self.model.aae_loss_func(pred, *yb)


class _AELoss:
    def __init__(self, model: ExplainableModel, recons_weight: float, adv_weight: float, corruption_cb: CorruptionCallback):
        self.model = model
        self.recons_weight = recons_weight
        self.adv_weight = adv_weight
        self.corruption_cb = corruption_cb

    def __call__(self, pred, *yb):
        clean_xb = self.corruption_cb.learn.clean_xb
        return self.model.denoising_ae_loss_func(clean_xb, self.recons_weight, self.adv_weight, pred, yb)


class _ClassifLoss:
    def __init__(self, model: ExplainableModel, recons_weight: float, class_weight: float, adv_weight: float):
        self.model = model
        self.recons_weight = recons_weight
        self.class_weight = class_weight
        self.adv_weight = adv_weight

    def __call__(self, pred, *yb):
        return self.model.classif_loss_func(
            pred, *yb, ADV_WEIGHT=self.adv_weight, RECONS_WEIGHT=self.recons_weight, CLASS_WEIGHT=self.class_weight
        )

In [ ]:
#| export
def train_xaaenet(
    model: ExplainableModel,
    dls: DataLoaders,
    *,
    epochs_adv: int = 35,
    epochs_ae: int = 50,
    epochs_classif: int = 30,
    ae_recons_weight: float = 0.4,
    ae_adv_weight: float = 0.6,
    classif_recons_weight: float = 0.599,
    classif_class_weight: float = 0.001,
    classif_adv_weight: float = 0.4,
    lr_max: float = 1e-4,
    lr_max_factor: float = 3.0,
    mask_ratio: float = 0.20,
    patch_size: int = 16,
    noise_std: float = 0.05,
    adv_low_threshold: float = 0.65,
    adv_high_threshold: float = 0.80,
    grad_accum: int = 4,
    patience: int = 10,
    models_dir: str | Path = "models",
    adv_fname: str = "xaaenet_adv",
    ae_fname: str = "xaaenet_ae",
    classif_fname: str = "xaaenet_classif",
    save_tsne: bool = True,
    tsne_max_points: int = 5000,
    tsne_perplexity: float = 30,
    save_pls: bool = True,
    pls_target_class: str | None = None,
    pls_max_points: int = 10_000,
    extract_latent: bool = True,
    latent_path: str | Path | None = None,
    show_latent_figures: bool | None = None,
) -> Learner:
    """Train an explainable xAAEnet model: adversarial, then autoencoder, then classifier.

    Parameters
    ----------
    model
        `AAE` or `EncoderWithAAEBlocks`.
    dls
        fastai `DataLoaders` with `(ImageBlock, CategoryBlock)`.
    ae_recons_weight, ae_adv_weight
        Weights for reconstruction and adversarial terms in the autoencoder phase.
    classif_recons_weight, classif_class_weight, classif_adv_weight
        Weights for the three terms in the classifier phase.
    adv_low_threshold, adv_high_threshold
        Validation-loss band driving `UnfreezeFcCritAdaptative` in the adversarial phase.
    save_tsne
        If True, save t-SNE figures after the adversarial and autoencoder phases.
    tsne_max_points, tsne_perplexity
        Subsample size and perplexity for those t-SNE plots.
    save_pls
        If True, after the classifier phase extract validation ``z``, show a supervised PLS biplot, save PNG.
    pls_target_class
        Name of the binary target level in `dls.vocab` for the PLS axis (default: second class).
    pls_max_points
        Cap on validation points for the PLS figure (default: 10_000).
    extract_latent
        If True, save stacked train+valid latent vectors to `latent_path` (or a default under `models_dir`).
    show_latent_figures
        If not False, display t-SNE / PLS figures inline when running inside a notebook (before saving PNG).
    """
    models_dir = Path(models_dir)
    models_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(default_device())
    lr_head = lr_max / lr_max_factor
    fit_cbs = [
        GradientAccumulation(n_acc=grad_accum),
        TrackerCallback(),
        EarlyStoppingCallback(min_delta=1e-4, patience=patience),
    ]

    print("Phase 1/3 — adversarial training")
    learn = Learner(dls, model, splitter=_aae_splitter, loss_func=_AAELoss(model), metrics=[LossAttrMetric("adv_loss")])
    learn.fit(
        epochs_adv,
        lr=slice(1e-6, 5e-5, lr_max),
        cbs=fit_cbs
        + [
            SaveModelCallback(fname=adv_fname),
            UnfreezeFcCritAdaptative(low_threshold=adv_low_threshold, high_threshold=adv_high_threshold),
        ],
    )
    _load_checkpoint(model, models_dir, adv_fname)
    if save_tsne:
        _tsne_validation_latent_after_phase(
            learn,
            models_dir / f"tsne_{adv_fname}.png",
            phase="adversarial",
            encoding_dims=getattr(model, "encoding_dims", None),
            max_points=tsne_max_points,
            perplexity=tsne_perplexity,
            show=show_latent_figures,
        )

    print("Phase 2/3 — autoencoder training")
    corruption_cb = CorruptionCallback([AddGaussianNoise(std=noise_std), RandomMasking(mask_ratio, patch_size)])
    learn = Learner(
        dls,
        model,
        loss_func=_AELoss(model, ae_recons_weight, ae_adv_weight, corruption_cb),
        metrics=[LossAttrMetric("recons_loss"), LossAttrMetric("adv_loss")],
        cbs=[corruption_cb],
    )
    corruption_cb.learn = learn
    learn.fit_one_cycle(epochs_ae, lr_max=lr_head, cbs=fit_cbs + [SaveModelCallback(fname=ae_fname)])
    _load_checkpoint(model, models_dir, ae_fname)
    if save_tsne:
        _tsne_validation_latent_after_phase(
            learn,
            models_dir / f"tsne_{ae_fname}.png",
            phase="autoencoder",
            encoding_dims=getattr(model, "encoding_dims", None),
            max_points=tsne_max_points,
            perplexity=tsne_perplexity,
            show=show_latent_figures,
        )

    print("Phase 3/3 — classifier training")
    learn = Learner(
        dls,
        model,
        loss_func=_ClassifLoss(model, classif_recons_weight, classif_class_weight, classif_adv_weight),
        metrics=[LossAttrMetric("adv_loss"), LossAttrMetric("recons_loss"), LossAttrMetric("classif_loss"), accuracy],
    )
    monitor = "valid_loss"
    learn.fit(
        epochs_classif,
        lr=lr_head,
        cbs=[
            GradientAccumulation(n_acc=grad_accum),
            TrackerCallback(monitor=monitor),
            EarlyStoppingCallback(min_delta=1e-4, patience=patience, monitor=monitor),
            SaveModelCallback(fname=classif_fname, monitor=monitor),
        ],
    )
    learn.load(classif_fname, with_opt=False, strict=False)

    torch.save(model.state_dict(), models_dir / f"{classif_fname}_final.pth")
    if save_pls:
        _pls_validation_latent_after_phase(
            learn,
            models_dir / f"pls_{classif_fname}.png",
            dls,
            phase="classifier",
            target_class=pls_target_class,
            encoding_dims=getattr(model, "encoding_dims", None),
            max_points=pls_max_points,
            show=show_latent_figures,
        )
    if extract_latent:
        z_parts = []
        for ds_idx in (0, 1):
            learn.z_valid = torch.tensor([]).to(default_device())
            learn.get_preds(ds_idx=ds_idx, cbs=[GetLatentSpace()])
            z_parts.append(learn.z_valid.clone())
        z_all = torch.vstack(z_parts)
        latent_path = latent_path or models_dir / f"{classif_fname}_z.pt"
        torch.save(z_all, latent_path)
    return learn